In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading the cleaned dataset

In [7]:
# Load the cleaned dataset from Google Drive
df_cleaned = pd.read_csv("df_cleaned.csv")

In [8]:
# Display the columns in the DataFrame
df_cleaned.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'Year',
       'review_length'],
      dtype='object')

In [9]:
# Display the shape of the DataFrame
df_cleaned.shape

(364154, 12)

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
import re

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

# Initialize lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Define contraction mappings
contractions_map = {
    "isn't": "is not",
    "can't": "cannot",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "won't": "will not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "shouldn't": "should not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "mustn't": "must not",
    "mightn't": "might not",
    "needn't": "need not"
}

def expand_contractions(text):
    """
    Expand contractions in the given text using a predefined mapping.
    """
    pattern = re.compile(r'\b(' + '|'.join(contractions_map.keys()) + r')\b')
    return pattern.sub(lambda x: contractions_map[x.group()], text)

def preprocess_text(text):
    """
    Preprocess text data:
    - Expand contractions.
    - Convert to lowercase.
    - Remove punctuation.
    - Tokenize and lemmatize words.
    """
    text = expand_contractions(text.lower())
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [lemmatizer.lemmatize(word) for word in word_tokenize(text) if word not in stop_words]
    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


- The code imports libraries for machine learning (e.g., `Logistic Regression`, `Naive Bayes`, and `Random Forest`) and natural language processing (e.g., `nltk` for tokenization, lemmatization, and stopword removal).  
- A `preprocess_text` function is defined to clean and standardize text by expanding contractions, converting to lowercase, removing punctuation, tokenizing, and lemmatizing while excluding stop words.  
- Contractions are expanded using the `contractions_map` dictionary (e.g., "isn't" → "is not") through the `expand_contractions` function, improving text clarity.  
- `TfidfVectorizer` transforms text data into numerical features for use in machine learning models.  
- Models like `Naive Bayes`, `Logistic Regression`, and `SVM`, along with techniques like `SMOTE` for oversampling and `BalancedRandomForestClassifier`, handle class imbalances for effective classification tasks.  

In [14]:

# Apply preprocessing to the 'Text' column
df_cleaned['Processed_Text'] = df_cleaned['Text'].apply(preprocess_text)

In [15]:
df_cleaned.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text', 'Year',
       'review_length', 'Processed_Text'],
      dtype='object')

Approach to Training Models:

- The next approach is as follows:
- The `X` variable contains the feature data, which is the **preprocessed text** from the `Processed_Text` column, and `y` contains the labels (`Score`), indicating whether the review is **positive** or **negative**.  
- The data is split into **training (80%)** and **testing (20%)** sets using `train_test_split`, ensuring the model can be trained on one portion and evaluated on unseen data.  
- A **TF-IDF Vectorizer** is initialized to transform text data into numerical feature vectors, with a maximum of 5000 features to capture the most important words.  
- The `fit_transform` method is applied to the training data to compute and transform it into TF-IDF features, while the `transform` method applies the learned transformation to the test data.  
- This process ensures that the textual data is converted into a format suitable for machine learning models while retaining the relative importance of words.  

# Multinomial NB with TF-IDF Vectorizer

In [16]:
# Multinomial NB with TF-IDF Vectorizer
# Split data into features and labels
X = df_cleaned['Processed_Text']
y = df_cleaned['Score']  # Already contains 'positive' or 'negative'

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train)
X_test_vectorized = tfidf_vectorizer.transform(X_test)

# Multinomial Naive Bayes Model
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

# Predict on test data
y_pred = model.predict(X_test_vectorized)

# Evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

Accuracy: 0.8766322033200148
Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.25      0.39     11531
    positive       0.88      0.99      0.93     61300

    accuracy                           0.88     72831
   macro avg       0.89      0.62      0.66     72831
weighted avg       0.88      0.88      0.85     72831



- **Accuracy:** 87.66%
- **Observations:**
  - Negative Reviews: Precision is high (90%), but recall is extremely low (25%), indicating that many negative reviews are misclassified.
  - Positive Reviews: High recall (99%) shows the model is very good at identifying positive reviews, but it heavily favors positive sentiment at the cost of negative reviews.
  - Macro Avg (F1-Score): 66% indicates poor balance between classes.
- **Conclusion:** This model is biased toward positive reviews and struggles to capture negative sentiment effectively.


# Multinomial NB with TF-IDF Vectorizer and SMOTE

In [17]:
# Multinomial NB with TF-IDF Vectorizer and SMOTE

# Split data into features and labels
X = df_cleaned['Processed_Text']
y = df_cleaned['Score']  # 'positive' or 'negative'

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = tfidf_vectorizer.fit_transform(X)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Train model on resampled data
model = MultinomialNB()
model.fit(X_train_resampled, y_train_resampled)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


Accuracy: 0.8670483722590655
Classification Report:
              precision    recall  f1-score   support

    negative       0.55      0.85      0.67     11531
    positive       0.97      0.87      0.92     61300

    accuracy                           0.87     72831
   macro avg       0.76      0.86      0.79     72831
weighted avg       0.90      0.87      0.88     72831



- **Accuracy:** 86.70%
- **Observations:**
  - Negative Reviews: Recall significantly improves to 85%, but precision drops to 55%, meaning many false positives for negative reviews.
  - Positive Reviews: Precision is very high (97%), but recall drops slightly (87%).
  - Macro Avg (F1-Score): 79%, showing better balance between classes than the previous model.
- **Conclusion:** The use of SMOTE helps balance the model for negative reviews but reduces precision, leading to overgeneralization of negatives.


# Logistic Regression with TF-IDF Vectorizer

In [18]:
# Logistic Regression with TF-IDF Vectorizer

# Split data into features and labels
X = df_cleaned['Processed_Text']
y = df_cleaned['Score']  # 'positive' or 'negative'

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train)
X_test_vectorized = tfidf_vectorizer.transform(X_test)

# Logistic Regression Model
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

# Predict on test data
y_pred = model.predict(X_test_vectorized)

# Evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


Accuracy: 0.9251417665554503
Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.66      0.74     11531
    positive       0.94      0.98      0.96     61300

    accuracy                           0.93     72831
   macro avg       0.89      0.82      0.85     72831
weighted avg       0.92      0.93      0.92     72831



- **Accuracy:** 92.51%
- **Observations:**
  - Negative Reviews: Precision (83%) and recall (66%) show decent performance for negative reviews.
  - Positive Reviews: Excellent precision (94%) and recall (98%), making it highly effective for positive reviews.
  - Macro Avg (F1-Score): 85%, showing a strong balance between classes.
- **Conclusion:** This model performs well for both positive and negative reviews with high accuracy and good balance.


# Logistic Regression with TF-IDF Vectorizer and SMOTE

In [19]:
# Logistic Regression with TF-IDF Vectorizer and SMOTE

# Split data into features and labels
X = df_cleaned['Processed_Text']
y = df_cleaned['Score']  # 'positive' or 'negative'

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = tfidf_vectorizer.fit_transform(X)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Logistic Regression Model
model = LogisticRegression()
model.fit(X_train_resampled, y_train_resampled)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


Accuracy: 0.8934794249701363
Classification Report:
              precision    recall  f1-score   support

    negative       0.62      0.87      0.72     11531
    positive       0.97      0.90      0.93     61300

    accuracy                           0.89     72831
   macro avg       0.79      0.88      0.83     72831
weighted avg       0.92      0.89      0.90     72831



- **Accuracy:** 89.35%
- **Observations:**
  - Negative Reviews: Recall improves significantly to 87%, but precision drops to 62%.
  - Positive Reviews: Precision is high (97%), but recall decreases slightly to 90%.
  - Macro Avg (F1-Score): 83%, showing decent balance but slightly worse performance than the Logistic Regression model without SMOTE.
- **Conclusion:** While SMOTE improves recall for negative reviews, it sacrifices precision, making the overall performance slightly worse.


# Random Forest with TF-IDF Vectorizer

In [20]:
# Random Forest with TF-IDF Vectorizer

# Split data into features and labels
X = df_cleaned['Processed_Text']
y = df_cleaned['Score']  # 'positive' or 'negative'

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train)
X_test_vectorized = tfidf_vectorizer.transform(X_test)

# Random Forest Model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vectorized, y_train)

# Predict on test data
y_pred = model.predict(X_test_vectorized)

# Evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)


Accuracy: 0.895250648762203
Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.39      0.54     11531
    positive       0.90      0.99      0.94     61300

    accuracy                           0.90     72831
   macro avg       0.89      0.69      0.74     72831
weighted avg       0.89      0.90      0.88     72831



- **Accuracy:** 89.52%
- **Observations:**
  - Negative Reviews: High precision (89%) but very low recall (39%), indicating many negative reviews are missed.
  - Positive Reviews: Excellent precision (90%) and recall (99%), favoring positive sentiment.
  - Macro Avg (F1-Score): 74%, showing poor balance compared to Logistic Regression.
- **Conclusion:** This model heavily favors positive reviews and performs poorly in identifying negative reviews.


# Conclusion

## Recommendation: Logistic Regression with TF-IDF Vectorizer (Without SMOTE)

- **Highest Accuracy:** Logistic Regression achieves the highest accuracy of **92.51%**, outperforming all other models in correctly classifying reviews.
- **Balanced Performance:** The model strikes a strong balance between **precision** and **recall** for both positive and negative reviews, as shown by a **macro average F1-score of 85%**.
  - **Negative Reviews:** Precision is **83%**, and recall is **66%**, which are better balanced than the other models.
  - **Positive Reviews:** Excellent precision (**94%**) and recall (**98%**) ensure that positive reviews are identified with high confidence.
- **No Overgeneralization Issues:** Unlike models with SMOTE, this Logistic Regression model does not sacrifice precision for recall. It avoids overgeneralizing the minority class (negative reviews) and maintains high specificity in its predictions.
- **Efficiency:** Logistic Regression is computationally efficient compared to ensemble models like Random Forest, making it suitable for large datasets like the Amazon Fine Food Reviews.
- **Robustness to Imbalance:** Despite class imbalance in the dataset, the model handles it effectively without requiring oversampling techniques like SMOTE, which can sometimes introduce noise or overfit the data.

### **Why Not Use SMOTE?**
- While SMOTE improves recall for the minority class (negative reviews), it does so at the expense of precision, which drops to **62%**.
- The Logistic Regression model without SMOTE maintains a better trade-off between precision and recall, resulting in a more reliable and interpretable classifier.

### **Conclusion:**
Logistic Regression with TF-IDF Vectorizer (without SMOTE) is the optimal choice for its combination of high accuracy, balanced class performance, computational efficiency, and robustness to class imbalance.
